## Project Solution (Kapitel 02)

Wie lässt sich das Gelernte aus Kapitel 02 an einem zusammenhängenden Beispiel anwenden? Dafür entsteht ein Model für ein Automobil. Es begleitet den ganzen Kurs: Am Ende jedes Kapitels wird es erweitert oder umgebaut, und später arbeitet es mit verwandten Models zusammen (Model Composition).

**Danach ist klar:**
- wie sich aus einer Field-Liste ein vollständiges Model mit Pflicht-Fields und Defaults ergibt,
- wie sich das Model prüfen lässt, indem `dict` und JSON-Text deserialisiert und das Ergebnis mit der erwarteten Ausgabe verglichen wird,
- warum `str | None` die passende Annotation für Fields mit dem Default `None` ist.

Voraussetzung: Kapitel 02, Notebooks `02` bis `10`, und die Aufgabenstellung in `11 - Project`.

Das Model `Automobile` enthält diese Fields:
- `manufacturer`: String, Pflicht, nicht nullable
- `series_name`: String, Pflicht, nicht nullable
- `type_`: String, Pflicht, nicht nullable
- `is_electric`: Boolean, Default `False`, nicht nullable
- `manufactured_date`: Datum, Pflicht, nicht nullable (als Typ dient `date` aus dem Modul `datetime`)
- `base_msrp_usd`: Float, Pflicht, nicht nullable
- `vin`: String, Pflicht, nicht nullable
- `number_of_doors`: Integer, Default `4`, nicht nullable
- `registration_country`: String, Default `None`
- `license_plate`: String, Default `None`

Funktioniert das Model auch? Das zeigt ein Test mit Deserialization und Serialization. Dafür liegen Beispieldaten bereit: Eingabedaten und die jeweils erwartete Ausgabe.

Getestet wird für beide Eingaben (`dict` und JSON-Text) in zwei Schritten:
1. Das Model entsteht durch Deserialization der Eingabedaten.
2. Die Serialization des Models als `dict` wird mit dem erwarteten Dictionary verglichen.

In [ ]:
# Eingabe als Python-`dict`: Alle Fields sind gesetzt, das Datum kommt noch als `str` an.
from datetime import date
from decimal import Decimal

data = {
    "manufacturer": "BMW",
    "series_name": "M4",
    "type_": "Convertible",
    "is_electric": False,
    "manufactured_date": "2023-01-01",
    "base_msrp_usd": 93_300,
    "vin": "1234567890",
    "number_of_doors": 2,
    "registration_country": "France",
    "license_plate": "AAA-BBB",
}

# Erwartete Ausgabe von `model_dump()`: Hier steht das Datum als `date`-Objekt.
data_expected_serialization = {
    'manufacturer': 'BMW',
    'series_name': 'M4',
    'type_': 'Convertible',
    'is_electric': False,
    'manufactured_date': date(2023,1,1),
    'base_msrp_usd': 93_300,
    'vin': '1234567890',
    'number_of_doors': 2,
    'registration_country': 'France',
    'license_plate': 'AAA-BBB',
}

In [ ]:
# Eingabe als JSON-Text, in dem einige Fields fehlen.
data_json = '''
{
    "manufacturer": "BMW",
    "series_name": "M4",
    "type_": "Convertible",
    "manufactured_date": "2023-01-01",
    "base_msrp_usd": 93300,
    "vin": "1234567890"
}
'''

# Erwartete Ausgabe von `model_dump()` für diese Eingabe.
data_json_expected_serialization = {
    'manufacturer': 'BMW',
    'series_name': 'M4',
    'type_': 'Convertible',
    'is_electric': False,
    'manufactured_date': date(2023, 1, 1),
    'base_msrp_usd': 93_300,
    'vin': '1234567890',
    'number_of_doors': 4,
    'registration_country': None,
    'license_plate': None,
}

## Solution

In [ ]:
# `date` ist der Typ für `manufactured_date`, `BaseModel` die Basisklasse des Models.
from datetime import date
from pydantic import BaseModel, ValidationError

Hinweis: `registration_country` und `license_plate` haben den Default `None` und sind deshalb mit `str | None` annotiert. Mit `str = None` würde die Definition ebenfalls durchlaufen, weil Pydantic Defaults nicht validiert. Sobald `None` aber explizit übergeben wird, etwa in `Automobile.model_validate(car.model_dump())`, schlägt die Validierung fehl, denn `None` ist kein `str`.

Das Zusammenspiel von `None`, Optional und Defaults steht in `07 - Nullable Fields` und `08 - Combining Nullable and Optional`.

In [ ]:
# Fields ohne Default sind Pflicht-Fields, Fields mit Default dürfen in der Eingabe fehlen.
class Automobile(BaseModel):
    manufacturer: str
    series_name: str
    type_: str
    is_electric: bool = False
    manufactured_date: date
    base_msrp_usd: float
    vin: str
    number_of_doors: int = 4
    registration_country: str | None = None
    license_plate: str | None = None

Wie wird aus dem `dict` eine `Automobile`-Instanz?

**Vorhersage:** Welchen Typ hat `manufactured_date` in `car`, obwohl in `data` ein `str` steht? Und welchen Wert hat `base_msrp_usd`, obwohl dort ein `int` übergeben wurde?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `model_validate` wandelt die Eingabewerte passend zu den Annotationen um (Coercion). Aus dem `str` `"2023-01-01"` wird ein `date`, aus dem `int` `93_300` ein `float` (`93300.0`). Beides zeigt die Ausgabe von `car`.

Kurz gesagt: Die Annotation bestimmt den Typ im Ergebnis, nicht der Typ der Eingabe.
</details>

In [ ]:
# `model_validate` deserialisiert das `dict` und liefert eine neue `Automobile`-Instanz.
car = Automobile.model_validate(data)
car

In [ ]:
# Der `assert` bleibt still, wenn beide Seiten gleich sind, sonst folgt ein `AssertionError`.
assert car.model_dump() == data_expected_serialization

In [ ]:
# `model_validate_json` liest den JSON-Text direkt, ein Umweg über `json.loads` ist nicht nötig. Fehlende Fields übernehmen ihren Default.
car = Automobile.model_validate_json(data_json)
car

In [ ]:
# Die Erwartung ist ein `dict`. Deshalb vergleicht der nächste `assert` mit `model_dump()` und nicht mit `model_dump_json()`, das einen `str` liefert.
type(data_json_expected_serialization)

In [ ]:
# Derselbe Vergleich wie oben: `model_dump()` gegen das erwartete `dict`.
assert car.model_dump() == data_json_expected_serialization

### Zusammenfassung

- Pflicht-Fields haben keinen Default, Fields mit Default dürfen in der Eingabe fehlen.
- `model_validate` deserialisiert ein `dict`, `model_validate_json` einen JSON-Text. Dabei wandelt Pydantic die Werte passend zu den Annotationen um.
- `model_dump()` liefert ein `dict`, das sich mit dem erwarteten Ergebnis vergleichen lässt. `model_dump_json()` liefert einen `str` und eignet sich dafür nicht.
- Fields mit dem Default `None` brauchen die Annotation `str | None`, sonst scheitert die erneute Validierung eines Dumps.

Weiter mit Kapitel `03 - Model Configuration`, Notebook `02 - Handling Extra Fields`.